In [ ]:
import os
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import folium
from folium.plugins import HeatMap, MarkerCluster, Fullscreen

import branca.colormap as cm


In [ ]:
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = f"acled_outputs_{RUN_TS}"
PLOTS_DIR = os.path.join(OUT_DIR, "plots")
MAPS_DIR = os.path.join(OUT_DIR, "maps")
TABLES_DIR = os.path.join(OUT_DIR, "tables")

os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(MAPS_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

def save_plot(name: str):
    safe = "".join(ch if ch.isalnum() or ch in "-_." else "_" for ch in name.strip())
    png_path = os.path.join(PLOTS_DIR, f"{safe}.png")
    pdf_path = os.path.join(PLOTS_DIR, f"{safe}.pdf")
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    print(f"✅ Saved plot: {png_path} + {pdf_path}")

print("Output folder:", OUT_DIR)


In [ ]:
CSV_PATH = "ACLED Data_2025-12-19-2.csv"
df = pd.read_csv(CSV_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df.head(3))


In [ ]:
df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")
df["year"] = df["event_date"].dt.year

df["fatalities"] = pd.to_numeric(df["fatalities"], errors="coerce").fillna(0).astype(float)

df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

display(df[["event_date", "year", "fatalities", "latitude", "longitude"]].head(5))


In [ ]:
TARGET_STATES = ["Gujarat", "Maharashtra", "Goa", "Madhya Pradesh", "Karnataka"]
df_states = df[df["admin1"].isin(TARGET_STATES)].copy()

print("Filtered rows:", len(df_states))
display(df_states[["event_date","admin1","location","event_type","sub_event_type","fatalities"]].head(10))


In [ ]:
sub_event_counts = (
    df_states.groupby("sub_event_type")
    .size()
    .sort_values(ascending=False)
    .reset_index(name="incidents")
)
sub_event_counts["share_%"] = 100 * sub_event_counts["incidents"] / sub_event_counts["incidents"].sum()

display(sub_event_counts)

plt.figure(figsize=(10,6))
sns.barplot(data=sub_event_counts.head(15), y="sub_event_type", x="incidents")
plt.title("Top 15 Sub-Event Types (Counts)")
plt.tight_layout()

save_plot("sub_event_types_top15_counts")
plt.show()

plt.figure(figsize=(10,6))
sns.barplot(data=sub_event_counts.head(15), y="sub_event_type", x="share_%")
plt.title("Top 15 Sub-Event Types (Share %)")
plt.tight_layout()

save_plot("sub_event_types_top15_share")
plt.show()


In [ ]:
fatalities_by_state = (
    df_states.groupby("admin1")["fatalities"]
    .sum()
    .reindex(TARGET_STATES)
    .fillna(0)
    .reset_index()
)
fatalities_by_state.columns = ["state", "fatalities"]
display(fatalities_by_state)

plt.figure(figsize=(8,5))
sns.barplot(data=fatalities_by_state, x="state", y="fatalities")
plt.title("Total Fatalities by State")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()

save_plot("fatalities_by_state")
plt.show()


In [ ]:
yearly = (
    df_states.dropna(subset=["year"])
    .groupby(["admin1","year"])
    .size()
    .reset_index(name="incidents")
    .sort_values(["admin1","year"])
)
display(yearly)

plt.figure(figsize=(10,5))
sns.lineplot(data=yearly, x="year", y="incidents", hue="admin1", marker="o")
plt.title("Yearly Incident Trend by State")
plt.grid(alpha=0.2)
plt.tight_layout()

save_plot("yearly_incident_trend_by_state")
plt.show()


In [ ]:
def top_actors_for_state(df_one_state, topk=20):
    a1 = df_one_state["actor1"].dropna().astype(str)
    a2 = df_one_state["actor2"].dropna().astype(str)
    actors = pd.concat([a1, a2], ignore_index=True)
    actors = actors[actors.str.strip() != ""]
    out = actors.value_counts().head(topk).reset_index()
    out.columns = ["actor", "incidents_involved"]
    return out

top_actors_tables = {}

for st in TARGET_STATES:
    top = top_actors_for_state(df_states[df_states["admin1"] == st], topk=20)
    top_actors_tables[st] = top
    print(f"\n### {st} — Top 20 Actors")
    display(top)

    plt.figure(figsize=(10,6))
    sns.barplot(data=top, y="actor", x="incidents_involved")
    plt.title(f"Top 20 Actors — {st}")
    plt.tight_layout()

    save_plot(f"top20_actors_{st}")
    plt.show()


In [ ]:
cols = ["event_date","admin1","location","event_type","sub_event_type","fatalities","notes"]
event_id_col = next((c for c in ["event_id_cnty","event_id_no_cnty"] if c in df_states.columns), None)
if event_id_col:
    cols = [event_id_col] + cols

top_fatal_events = (
    df_states[cols]
    .sort_values(["admin1","fatalities"], ascending=[True, False])
    .groupby("admin1")
    .head(3)
)

for st in TARGET_STATES:
    print(f"\n### {st} — Top 3 Most Fatal Events")
    display(top_fatal_events[top_fatal_events["admin1"] == st].reset_index(drop=True))


In [ ]:
city_counts = (
    df_states.groupby(["location","admin1"])
    .size()
    .reset_index(name="incidents")
    .sort_values("incidents", ascending=False)
    .head(15)
)
display(city_counts)

plt.figure(figsize=(10,6))
labels = city_counts["location"] + " (" + city_counts["admin1"] + ")"
plt.barh(labels[::-1], city_counts["incidents"][::-1])
plt.title("Top 15 Locations by Incident Count")
plt.xlabel("Incidents")
plt.tight_layout()

save_plot("top15_locations_by_incidents")
plt.show()


In [ ]:
df_map = df_states.dropna(subset=["latitude","longitude"]).copy()
df_map = df_map.dropna(subset=["latitude","longitude"])

print("Map points available:", len(df_map))
display(df_map[["admin1","location","sub_event_type","fatalities","latitude","longitude"]].head(5))


In [ ]:
MAP_CENTER = [20.5, 78.5]
ZOOM = 5

STATE_COLORS = {
    "Gujarat": "#1f77b4",
    "Maharashtra": "#ff7f0e",
    "Goa": "#2ca02c",
    "Madhya Pradesh": "#d62728",
    "Karnataka": "#9467bd"
}

def clamp(v, lo, hi):
    return max(lo, min(hi, v))

def radius_from_fatalities(f):
    try:
        f = float(f)
    except:
        f = 0
    return clamp(2 + np.sqrt(f) * 2, 2, 18)

max_fat = float(df_map["fatalities"].max()) if len(df_map) else 1.0
FATAL_COLORMAP = cm.linear.YlOrRd_09.scale(0, max_fat)
FATAL_COLORMAP.caption = "Fatalities (color scale)"


In [ ]:
m_heat = folium.Map(location=MAP_CENTER, zoom_start=ZOOM, tiles="CartoDB positron")
Fullscreen(position="topright").add_to(m_heat)

HeatMap(
    df_map[["latitude","longitude"]].values.tolist(),
    radius=12,
    blur=18,
    min_opacity=0.3
).add_to(m_heat)

folium.TileLayer("OpenStreetMap", name="OSM (Detailed)", show=False).add_to(m_heat)
folium.TileLayer("CartoDB dark_matter", name="Dark Mode", show=False).add_to(m_heat)

folium.LayerControl(collapsed=False).add_to(m_heat)
m_heat


In [ ]:
m_states = folium.Map(location=MAP_CENTER, zoom_start=ZOOM, tiles="CartoDB positron")
Fullscreen(position="topright").add_to(m_states)

# Overall heatmap ON
fg_all = folium.FeatureGroup(name="All States — Heatmap", show=True)
HeatMap(df_map[["latitude","longitude"]].values.tolist(), radius=12, blur=18, min_opacity=0.3).add_to(fg_all)
fg_all.add_to(m_states)

for st in TARGET_STATES:
    df_st = df_map[df_map["admin1"] == st]
    color = STATE_COLORS.get(st, "#333333")

    # Heatmap per state OFF
    fg_heat = folium.FeatureGroup(name=f"{st} — Heatmap", show=False)
    if len(df_st):
        HeatMap(df_st[["latitude","longitude"]].values.tolist(), radius=12, blur=18, min_opacity=0.3).add_to(fg_heat)
    fg_heat.add_to(m_states)

    # Points per state OFF (sample)
    fg_pts = folium.FeatureGroup(name=f"{st} — Points (sample)", show=False)
    if len(df_st):
        sample_n = min(len(df_st), 1200)
        for _, r in df_st.sample(sample_n, random_state=1).iterrows():
            popup = folium.Popup(
                f"""
                <b>State:</b> {r.get('admin1','')}<br>
                <b>Location:</b> {r.get('location','')}<br>
                <b>Sub-event:</b> {r.get('sub_event_type','')}<br>
                <b>Fatalities:</b> {r.get('fatalities',0)}
                """,
                max_width=350
            )
            folium.CircleMarker(
                location=[r["latitude"], r["longitude"]],
                radius=3,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.6,
                popup=popup
            ).add_to(fg_pts)
    fg_pts.add_to(m_states)

folium.TileLayer("CartoDB dark_matter", name="Dark Mode", show=False).add_to(m_states)
folium.LayerControl(collapsed=False).add_to(m_states)
m_states


In [ ]:
m_sub = folium.Map(location=MAP_CENTER, zoom_start=ZOOM, tiles="CartoDB positron")
Fullscreen(position="topright").add_to(m_sub)

top_sub_events = sub_event_counts["sub_event_type"].head(8).tolist()
print("Top sub-events used:", top_sub_events)

SUB_COLORS = ["#e41a1c","#377eb8","#4daf4a","#984ea3","#ff7f00","#ffff33","#a65628","#f781bf"]

for se, col in zip(top_sub_events, SUB_COLORS):
    df_se = df_map[df_map["sub_event_type"] == se]

    fg_heat = folium.FeatureGroup(name=f"{se} — Heatmap", show=False)
    if len(df_se):
        HeatMap(df_se[["latitude","longitude"]].values.tolist(), radius=12, blur=18, min_opacity=0.25).add_to(fg_heat)
    fg_heat.add_to(m_sub)

    fg_pts = folium.FeatureGroup(name=f"{se} — Points (sample)", show=False)
    if len(df_se):
        sample_n = min(len(df_se), 900)
        for _, r in df_se.sample(sample_n, random_state=1).iterrows():
            popup = folium.Popup(
                f"""
                <b>State:</b> {r.get('admin1','')}<br>
                <b>Location:</b> {r.get('location','')}<br>
                <b>Sub-event:</b> {r.get('sub_event_type','')}<br>
                <b>Fatalities:</b> {r.get('fatalities',0)}
                """,
                max_width=350
            )
            folium.CircleMarker(
                location=[r["latitude"], r["longitude"]],
                radius=3,
                color=col,
                fill=True,
                fill_color=col,
                fill_opacity=0.55,
                popup=popup
            ).add_to(fg_pts)
    fg_pts.add_to(m_sub)

folium.TileLayer("CartoDB dark_matter", name="Dark Mode", show=False).add_to(m_sub)
folium.LayerControl(collapsed=False).add_to(m_sub)
m_sub


In [ ]:
m_severity = folium.Map(location=MAP_CENTER, zoom_start=ZOOM, tiles="CartoDB positron")
Fullscreen(position="topright").add_to(m_severity)

df_sev = df_map.copy()
if len(df_sev) > 8000:
    df_sev = df_sev.sample(8000, random_state=2)

fg = folium.FeatureGroup(name="Fatality severity — Points", show=True)

for _, r in df_sev.iterrows():
    f = float(r.get("fatalities", 0))
    col = FATAL_COLORMAP(f)

    popup = folium.Popup(
        f"""
        <b>State:</b> {r.get('admin1','')}<br>
        <b>Location:</b> {r.get('location','')}<br>
        <b>Sub-event:</b> {r.get('sub_event_type','')}<br>
        <b>Fatalities:</b> {f}
        """,
        max_width=350
    )

    folium.CircleMarker(
        location=[r["latitude"], r["longitude"]],
        radius=radius_from_fatalities(f),
        color=col,
        fill=True,
        fill_color=col,
        fill_opacity=0.65,
        weight=1,
        popup=popup
    ).add_to(fg)

fg.add_to(m_severity)

FATAL_COLORMAP.add_to(m_severity)

folium.TileLayer("CartoDB dark_matter", name="Dark Mode", show=False).add_to(m_severity)
folium.LayerControl(collapsed=False).add_to(m_severity)
m_severity


In [ ]:
pivot = (
    df_states.pivot_table(
        index="admin1",
        columns="sub_event_type",
        values="fatalities",
        aggfunc="size",
        fill_value=0
    ).reindex(TARGET_STATES)
)

top_cols = pivot.sum(axis=0).sort_values(ascending=False).head(15).index
pivot15 = pivot[top_cols]

display(pivot15)

plt.figure(figsize=(14,5))
sns.heatmap(pivot15, cmap="Reds", linewidths=0.3)
plt.title("Heatmap: State × Sub-Event Type (Top 15 Sub-Events)")
plt.tight_layout()

save_plot("state_x_subevent_heatmap_top15")
plt.show()


In [ ]:
# Tables
sub_event_counts.to_csv(os.path.join(TABLES_DIR, "sub_event_counts.csv"), index=False)
fatalities_by_state.to_csv(os.path.join(TABLES_DIR, "fatalities_by_state.csv"), index=False)
yearly.to_csv(os.path.join(TABLES_DIR, "yearly_incidents_by_state.csv"), index=False)
city_counts.to_csv(os.path.join(TABLES_DIR, "top15_cities.csv"), index=False)
pivot15.reset_index().to_csv(os.path.join(TABLES_DIR, "state_x_subevent_top15.csv"), index=False)

# Maps (HTML)
m_heat.save(os.path.join(MAPS_DIR, "map_01_heat_incidents_pretty.html"))
m_states.save(os.path.join(MAPS_DIR, "map_02_by_state_layers.html"))
m_sub.save(os.path.join(MAPS_DIR, "map_03_by_subevent_layers.html"))
m_severity.save(os.path.join(MAPS_DIR, "map_04_fatality_severity.html"))

# README
with open(os.path.join(OUT_DIR, "README.txt"), "w") as f:
    f.write(
        "ACLED EXPORTS\n"
        "============\n\n"
        "plots/  -> static plots (PNG + PDF)\n"
        "maps/   -> interactive folium maps (HTML)\n"
        "tables/ -> csv tables\n\n"
        "Open HTML maps in a browser for interactivity.\n"
    )

print("✅ Export complete!")
print("📁 Saved to:", OUT_DIR)
print(" - Plots:", PLOTS_DIR)
print(" - Maps :", MAPS_DIR)
print(" - Tables:", TABLES_DIR)
